# 02 Data model

**Graduates to:** `hub/src/hudhud_hub/db.py`

The hub's whole state lives in one SQLite file. Open
`hub/src/hudhud_hub/migrations/0001_init.sql` next to this notebook and read
it: it is the schema from spec section 7.1.

## Four ideas

**Migrations.** The schema changes over time. Each change is a numbered file
(`0001_init.sql`, `0002_vectors.sql`, ...). A `schema_version` table records
which numbers have run. At startup the hub applies every file with a higher
number, in order. Each file runs in **one transaction together with its
`schema_version` row**, so a migration either fully happens or not at all.
Without that, a crash halfway leaves a database that is neither version 1
nor version 2.

**Transactions in `sqlite3`.** Opening with `isolation_level=None` means
Python never starts transactions behind your back; you write `BEGIN` and
`COMMIT` yourself. `executescript` runs many statements at once; if one fails,
it raises and leaves the transaction open, so you must `ROLLBACK`.

**ULIDs.** Highlight ids are made on phones that may be offline, so they
cannot come from an auto-increment counter. A ULID is 128 bits: 48 bits of
millisecond timestamp plus 80 random bits, written as 26 Crockford base-32
characters. Two devices never collide, and ids sort by creation time.

**Soft deletes.** Deleting a row would let an offline device that never heard
about the delete push it back to life. Instead, set `deleted = 1` (a
"tombstone") and filter it out when reading. Sync (notebook 08) depends on
this.

## Pragmas worth knowing

- `PRAGMA foreign_keys = ON`: SQLite ignores `REFERENCES` unless you turn
  this on, per connection.
- `PRAGMA journal_mode = WAL`: readers do not block the writer. This matters
  once the API and the indexer thread share the file. It does not apply to
  `:memory:`.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-or-later
import os
import re
import sqlite3
import time
from datetime import UTC, datetime
from pathlib import Path

import ipytest
from ulid import ULID

ipytest.autoconfig(raise_on_error=os.environ.get("HUDHUD_NB_CHECK") == "1")

MIGRATIONS = Path("../hub/src/hudhud_hub/migrations")
SOFT_DELETABLE = {"books", "highlights"}
MIGRATION_RE = re.compile(r"^(\d{4})_[\w-]+\.sql$")

## Your code

In [ ]:
def connect(path: str | Path) -> sqlite3.Connection:
    """Open with isolation_level=None and Row factory; foreign keys on; WAL unless ':memory:'."""
    raise NotImplementedError


def new_ulid() -> str:
    """A new ULID as a 26-character string (python-ulid: str(ULID()))."""
    raise NotImplementedError


def utc_now() -> str:
    """Current UTC time as 'YYYY-MM-DDTHH:MM:SS.mmmZ'."""
    raise NotImplementedError


def schema_version(conn: sqlite3.Connection) -> int:
    """Create schema_version(version INTEGER PRIMARY KEY, applied_at TEXT NOT NULL)
    if missing; return the highest applied version, or 0."""
    raise NotImplementedError


def apply_migrations(conn: sqlite3.Connection, migrations_dir: Path) -> list[int]:
    """Apply every NNNN_name.sql above the current version, in order.

    Each file plus its schema_version row runs inside one BEGIN ... COMMIT via
    executescript. On error: ROLLBACK if a transaction is open, then re-raise.
    Raise ValueError if two files share a version number.
    Return the versions applied by this call.
    """
    raise NotImplementedError


def insert_book(conn: sqlite3.Connection, book: dict, now: str) -> bool:
    """Insert a book (keys: id, title, author, language, format, file_size,
    cover_path), with added_at = updated_at = now. If the id exists and is
    deleted, restore it instead (deleted = 0, updated_at = now): importing the
    same file again is how you get a deleted book back. Use
    INSERT ... ON CONFLICT (id) DO UPDATE ... WHERE books.deleted = 1.
    True if a row was inserted or restored, False if a live copy already existed."""
    raise NotImplementedError


def list_books(conn: sqlite3.Connection) -> list[sqlite3.Row]:
    """Books that are not deleted, newest first, then by title."""
    raise NotImplementedError


def soft_delete(conn: sqlite3.Connection, table: str, entity_id: str, ts: str) -> bool:
    """Set deleted = 1 and updated_at = ts. Only tables in SOFT_DELETABLE are
    allowed (ValueError otherwise; table names cannot be ? placeholders).
    True if a live row was tombstoned, False if missing or already deleted."""
    raise NotImplementedError

## Tests

In [ ]:
def book(book_id="a" * 64, title="Masnavi"):
    return {"id": book_id, "title": title, "author": "Rumi", "language": "fa",
            "format": "epub", "file_size": 123, "cover_path": None}


def migrated():
    conn = connect(":memory:")
    apply_migrations(conn, MIGRATIONS)
    return conn

In [ ]:
%%ipytest -q
import pytest


def test_connect_pragmas(tmp_path):
    conn = connect(tmp_path / "hub.db")
    assert conn.execute("PRAGMA foreign_keys").fetchone()[0] == 1
    assert conn.execute("PRAGMA journal_mode").fetchone()[0] == "wal"
    assert conn.row_factory is sqlite3.Row


def test_fresh_database_gets_every_table():
    conn = connect(":memory:")
    assert apply_migrations(conn, MIGRATIONS) == [1]
    tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    assert {"books", "progress", "highlights", "devices", "changes", "chunks",
            "schema_version"} <= tables


def test_migrations_are_idempotent():
    conn = migrated()
    assert apply_migrations(conn, MIGRATIONS) == []
    assert schema_version(conn) == 1


def test_failed_migration_rolls_back(tmp_path):
    (tmp_path / "0001_init.sql").write_text((MIGRATIONS / "0001_init.sql").read_text("utf-8"))
    (tmp_path / "0002_broken.sql").write_text("CREATE TABLE half (x);\nCREATE TABLE broken (")
    conn = connect(":memory:")
    with pytest.raises(sqlite3.Error):
        apply_migrations(conn, tmp_path)
    assert schema_version(conn) == 1
    assert conn.execute("SELECT name FROM sqlite_master WHERE name = 'half'").fetchone() is None
    assert not conn.in_transaction


def test_duplicate_versions_rejected(tmp_path):
    (tmp_path / "0001_a.sql").write_text("CREATE TABLE a (x);")
    (tmp_path / "0001_b.sql").write_text("CREATE TABLE b (x);")
    with pytest.raises(ValueError):
        apply_migrations(connect(":memory:"), tmp_path)


def test_foreign_keys_enforced():
    conn = migrated()
    with pytest.raises(sqlite3.IntegrityError):
        conn.execute("INSERT INTO progress VALUES ('missing-book', 'dev', 'cfi', 0.5, ?)",
                     (utc_now(),))


def test_fraction_must_be_between_0_and_1():
    conn = migrated()
    insert_book(conn, book(), utc_now())
    with pytest.raises(sqlite3.IntegrityError):
        conn.execute("INSERT INTO progress VALUES (?, 'dev', 'cfi', 1.5, ?)",
                     (book()["id"], utc_now()))


def test_ulids_are_unique_and_time_sortable():
    first = new_ulid()
    time.sleep(0.002)
    later = [new_ulid() for _ in range(100)]
    assert len(set(later)) == 100
    assert all(len(u) == 26 and re.fullmatch(r"[0-9A-HJKMNP-TV-Z]{26}", u) for u in later)
    assert all(first < u for u in later)


def test_utc_now_format():
    assert re.fullmatch(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\d\.\d{3}Z", utc_now())


def test_insert_book_is_idempotent():
    conn = migrated()
    assert insert_book(conn, book(), "2026-10-08T10:00:00.000Z") is True
    assert insert_book(conn, book(title="Changed"), "2026-10-08T11:00:00.000Z") is False
    rows = list_books(conn)
    assert len(rows) == 1 and rows[0]["title"] == "Masnavi"


def test_soft_delete_hides_once():
    conn = migrated()
    insert_book(conn, book(), utc_now())
    assert soft_delete(conn, "books", book()["id"], utc_now()) is True
    assert soft_delete(conn, "books", book()["id"], utc_now()) is False
    assert list_books(conn) == []
    assert conn.execute("SELECT deleted FROM books").fetchone()[0] == 1


def test_reimporting_a_deleted_book_restores_it():
    conn = migrated()
    insert_book(conn, book(), "2026-10-08T10:00:00.000Z")
    soft_delete(conn, "books", book()["id"], "2026-10-08T11:00:00.000Z")
    assert insert_book(conn, book(), "2026-10-08T12:00:00.000Z") is True
    rows = list_books(conn)
    assert len(rows) == 1 and rows[0]["updated_at"] == "2026-10-08T12:00:00.000Z"


def test_soft_delete_rejects_other_tables():
    conn = migrated()
    for table in ["devices", "books; DROP TABLE books"]:
        with pytest.raises(ValueError):
            soft_delete(conn, table, "x", utc_now())

## Graduate

1. Create `hub/src/hudhud_hub/db.py` from the imports (minus `ipytest`),
   constants and your functions. In the module, default the migrations
   folder to `Path(__file__).parent / "migrations"`.
2. Move the fixture and test cells to `hub/tests/test_db.py` with
   `from hudhud_hub.db import *`, and point `MIGRATIONS` at the package folder.
3. `uv run --project hub pytest hub/tests/test_db.py -q`